# extra-deep - Kaggle runner

**This notebook is deliberately thin. It contains NO experiment logic.**
Everything scientific lives in the repository, under version control. This notebook
only: clones, installs, points DATA_ROOT at the attached dataset, calls ONE script,
and copies `artifacts/` back out.

If you find yourself editing an experiment parameter here, stop: edit `configs/*.yaml`
in the repository, push, and re-run this notebook.

See `docs/KAGGLE_SETUP.md` for how to attach the dataset and set the accelerator.

---
**Session boundary.** Kaggle sessions end after ~12 h. Every script is resumable via
`artifacts/registry.jsonl`, so the recipe is: run, copy `artifacts/` out at the end
(last cell), commit it, and on the next session restore it before running again.


## 1. Clone the repository


In [ ]:
# Set REPO_URL to your remote. A private repo needs a token in the URL or a Kaggle Secret.
REPO_URL = 'https://github.com/<your-user>/extra-deep.git'
BRANCH   = 'main'

import os, shutil, subprocess, sys
WORK = '/kaggle/working/extra-deep'
if os.path.exists(WORK):
    shutil.rmtree(WORK)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,WORK], check=True)
os.chdir(WORK)
print('cwd:', os.getcwd())
subprocess.run(['git','log','-1','--oneline'], check=False)


## 2. Install dependencies

Kaggle's base image already carries a CUDA build of torch. Let the pre-installed
wheel win rather than forcing a reinstall of the CPU pin from `requirements.txt`.


In [ ]:
!pip install -q --upgrade-strategy only-if-needed -r requirements.txt

import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Settings -> Accelerator -> GPU T4 x2 (or P100).')


## 3. Point DATA_ROOT at the attached dataset

Dataset slug `raihanakirar/srp-dyna-card`, mounted at `/kaggle/input/srp-dyna-card`.
Note the `dataset/` wrapper directory inside the mount. Nothing is hardcoded: the
path comes from `configs/data.yaml` and is overridden here by the environment
variable, so the same code runs unchanged locally and on a Raspberry Pi.


In [ ]:
import os
os.environ['SRPCARD_DATA_ROOT'] = '/kaggle/input/srp-dyna-card/dataset'

# Restore artifacts/ from a previous session if you attached it as a dataset.
# Set to None on the very first run.
RESUME_FROM = None   # e.g. '/kaggle/input/extra-deep-artifacts'
if RESUME_FROM:
    import shutil, glob
    for src in glob.glob(RESUME_FROM + '/*'):
        shutil.copy2(src, 'artifacts/')
    print('restored:', os.listdir('artifacts'))

!ls -1 $SRPCARD_DATA_ROOT | head -20
!echo '--- images:' && find $SRPCARD_DATA_ROOT -type f | wc -l


## 4. Run ONE script

Uncomment exactly one line. Run them in this order across sessions:

| script | what | runs |
|---|---|---|
| `00_build_folds.py` | index, dev split, folds. **Verifies committed artefacts.** | - |
| `01_complete_medium_grid.py` | 8 missing medium configs, legacy protocol | 8 |
| `02_lr_sweep_baselines.py` | baseline lr sweep | 6 |
| `03_run_cv.py` | the main experiment | 75 |
| `04_run_ablation.py` | class-weight ablation | 15 |
| `05_learning_curve.py` | learning curve | 225 |
| `06_export_figures.py` | figures and tables | - |

Scripts 01 and 02 write back into `configs/arms.yaml`. That file must be committed
before 03 runs, or 03 will use stale hyperparameters.

Every script is resumable: re-running skips what is already in the registry.


In [ ]:
# !python scripts/00_build_folds.py
# !python scripts/01_complete_medium_grid.py
# !python scripts/02_lr_sweep_baselines.py
!python scripts/03_run_cv.py --quiet
# !python scripts/04_run_ablation.py --quiet
# !python scripts/05_learning_curve.py --quiet
# !python scripts/06_export_figures.py


## 5. Copy artifacts/ back out

**Run this cell even if the script above was interrupted.** The registry is
append-only and flushed after every run, so a partial `artifacts/` is still worth
keeping - it is exactly what lets the next session resume.

Everything under `/kaggle/working/artifacts_out/` is downloadable from the notebook
output panel. Commit `registry.jsonl` and any changed `configs/arms.yaml`.


In [ ]:
import shutil, os, glob
OUT = '/kaggle/working/artifacts_out'
os.makedirs(OUT, exist_ok=True)
for src in glob.glob('artifacts/**/*', recursive=True):
    if os.path.isfile(src):
        dst = os.path.join(OUT, os.path.relpath(src, 'artifacts'))
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)

shutil.copy2('configs/arms.yaml', os.path.join(OUT, 'arms.yaml'))

for root, _, files in os.walk(OUT):
    for f in sorted(files):
        p = os.path.join(root, f)
        print('%9.1f KB  %s' % (os.path.getsize(p)/1024, os.path.relpath(p, OUT)))

shutil.make_archive('/kaggle/working/artifacts_bundle', 'zip', OUT)
print('
bundle: /kaggle/working/artifacts_bundle.zip')


## 6. Progress check

How much is done, and how much is left.


In [ ]:
import json, collections
counts = collections.Counter()
try:
    with open('artifacts/registry.jsonl', encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if line:
                r = json.loads(line)
                counts[(r.get('script'), r.get('arm'))] += 1
except FileNotFoundError:
    print('no registry yet')
for (script, arm), n in sorted(counts.items()):
    print('%-26s %-20s %3d' % (script, arm, n))
print('total records:', sum(counts.values()))
